In [ ]:
import os

# 파일 업로드 필요 없이, 시스템에 직접 인증 정보를 주입하는 방식입니다.
os.environ["KAGGLE_USERNAME"] = "ixddddl"  # 여기에 본인 카글 ID 입력
os.environ["KAGGLE_KEY"] = (
    "0ad9cb51b9939984acea86867f31c715"  # 여기에 본인 API Key 입력
)

In [ ]:
%pip install -q kagglehub

import kagglehub

# Download latest version
path = kagglehub.competition_download("ai12-level1-project")
print("Path to competition files:", path)

In [ ]:
# 한글 폰트 설치
!apt-get install -y fonts-nanum > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import os

DATA_DIR = os.path.join(path, "sprint_ai_project1_data")

print(DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_ANN = os.path.join(DATA_DIR, "train_annotations")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test_images")

print(f"tran 이미지 개수: {len(os.listdir(TRAIN_IMAGE_DIR))}")
print(f"test 이미지 개수: {len(os.listdir(TEST_IMAGE_DIR))}")
print(f"train annotation 폴더 목록: {os.listdir(TRAIN_ANN)}")

In [ ]:
print(os.listdir(TRAIN_ANN))

In [ ]:
import os

json_paths = []

for root, dirs, files in os.walk(TRAIN_ANN):
    for file in files:
        if file.endswith(".json"):
            json_paths.append(os.path.join(root, file))

print(f"JSON 파일 개수 : {len(json_paths)}")
print(json_paths[:5])

In [ ]:
import os
import json

all_images = []
all_annotations = []
all_categories = []

for root, dirs, files in os.walk(TRAIN_ANN):
    for file in files:
        if file.endswith(".json"):
            path = os.path.join(root, file)

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            all_images.extend(data["images"])
            all_annotations.extend(data["annotations"])
            all_categories.extend(data["categories"])

images_df = pd.DataFrame(all_images).drop_duplicates("id")
annotations_df = pd.DataFrame(all_annotations)
categories_df = pd.DataFrame(all_categories).drop_duplicates("id")

print(images_df.shape)
print(annotations_df.shape)
print(categories_df.shape)

In [ ]:
print(categories_df["name"].unique())

In [ ]:
import pandas as pd

images_df = pd.DataFrame(all_images)
images_df = images_df.drop_duplicates(subset="id")

print(images_df.shape)
images_df.head()

In [ ]:
selected_image = images_df.iloc[0]

print(selected_image["file_name"])
print(selected_image["id"])

In [ ]:
import os

img_path = os.path.join(TRAIN_IMAGE_DIR, selected_image["file_name"])

print(img_path)

In [ ]:
pill_count = annotations_df.groupby("image_id").size()

print(pill_count.describe())
print(pill_count.value_counts())

In [ ]:
plt.figure(figsize=(6, 4))

pill_count.value_counts().sort_index().plot(kind="bar")

plt.xlabel("Number of Pills")
plt.ylabel("Number of Images")
plt.title("Pills per Image")
plt.show()

In [ ]:
merged_df = annotations_df.merge(
    images_df[["id", "camera_la"]], left_on="image_id", right_on="id"
)

merged_df["camera_la"].value_counts()

In [ ]:
class_count = annotations_df["category_id"].value_counts().reset_index()

class_count.columns = ["category_id", "count"]

class_count = class_count.merge(
    categories_df[["id", "name"]], left_on="category_id", right_on="id", how="left"
)

class_count = class_count.sort_values("count", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.bar(class_count["name"], class_count["count"])

plt.xticks(rotation=90)
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()

In [ ]:
plt.scatter(images_df["width"], images_df["height"])
plt.show()

In [ ]:
images_df["drug_shape"].value_counts()

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# ===== 시작 인덱스 설정 =====
start_idx = 64  # 0, 16, 32, 48 ... 바꾸면서 확인
n_images = 16

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for i in range(n_images):
    idx = start_idx + i

    if idx >= len(images_df):
        break

    # 이미지 정보
    img_info = images_df.iloc[idx]

    img_path = os.path.join(TRAIN_IMAGE_DIR, img_info["file_name"])

    img = Image.open(img_path)

    axes[i].imshow(img)

    # 해당 이미지의 annotation
    image_annotations = annotations_df[annotations_df["image_id"] == img_info["id"]]

    # bbox 그리기
    for _, row in image_annotations.iterrows():
        x, y, w, h = row["bbox"]

        rect = patches.Rectangle(
            (x, y), w, h, linewidth=2, edgecolor="red", facecolor="none"
        )

        axes[i].add_patch(rect)

    # 제목
    axes[i].set_title(
        f"idx : {idx}\n"
        f"id : {img_info['id']}\n"
        f"box : {len(image_annotations)}\n"
        f"{img_info['file_name']}",
        fontsize=7,
    )

    axes[i].axis("off")

# 빈 subplot 제거
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 테스트 이미지 출력
import os
import math
from PIL import Image
import matplotlib.pyplot as plt


img_list = sorted(os.listdir(TEST_IMAGE_DIR))[:20]

cols = 5
rows = math.ceil(len(img_list) / cols)

plt.figure(figsize=(15, 3 * rows))

for i, img_name in enumerate(img_list):
    img = Image.open(os.path.join(TEST_IMAGE_DIR, img_name))

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(img_name, fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()